# NN 3 parameters case

In [1]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam,Nadam,Adamax
from ann_functions3D import getModel, kCrossVal, transfBestparam
from time import perf_counter
import pandas
import pickle
import os

seed = 7
np.random.seed(seed)

c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
########################     PREPARATION      ##########################
HF_data = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/HF_num_data.txt").astype(int) #list of number of HF data
Nlf_models = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/N_bases.txt").astype(int)[6:8]
HF_data_str = [str(num) for num in HF_data]
r2_HF_df = pandas.DataFrame(index=HF_data_str)             #dataframe which stores HF R^2
r2_LF_df = pandas.DataFrame()                              #dataframe which stores LF R^2
r2_Lin_df = pandas.DataFrame(index=HF_data_str)            #dataframe which stores Lin R^2

mse_HF_df = pandas.DataFrame(index=HF_data_str)            #dataframe which stores HF MSE
mse_LF_df = pandas.DataFrame()                             #dataframe which stores LF MSE
mse_Lin_df = pandas.DataFrame(index=HF_data_str)           #dataframe which stores Lin MSE

U_HF_list  = []
U_LF_list  = []
U_Lin_list = []

In [3]:
#########################     TRAIN SET      ##########################
mu_train_LF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_LF.txt")
Nlf = np.size(mu_train_LF)    # number of low-fidelity data to TRAIN the NN

U_lf_train_full = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Ulf_train.txt")[:,6:8]

NepoLF = 3000   # number of epochs for first NN: NN_LF
NepoLin = 1500  # number of epochs for second NN: NN_HF
NepoHF = 3000   # number of epochs for third NN: NN_HF

# NB su codice di paolo il numero di steps è diverso wrt hyperparams research. PErchè?

In [4]:
#########################     TEST SET      ##########################
mu_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/mu_test.txt")
N_test = np.size(mu_test)

U_lf_test_full = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Ulf_test.txt")[:,6:8]

# ADD NOISE

permutation = np.random.permutation(len(mu_train_LF))
mu_train_LF=mu_train_LF[permutation,:][0:10,:]

noise_stddev = np.mean(mu_train_LF,axis=0)*0.1
noise = np.random.normal(0, noise_stddev, np.shape(mu_train_LF))
mu_train_LF=mu_train_LF+noise 

Nlf = np.size(mu_train_LF)  #number of low-fidelity data to TRAIN the NN

U_lf_train_full=U_lf_train_full[permutation][0:10,:]

noise_stddev = 0.005
noise = np.random.normal(0, noise_stddev, np.shape(U_lf_train_full))
U_lf_train_full=U_lf_train_full+noise

In [5]:
#Input
mu_max = np.max(mu_test)
mu_min = np.min(mu_test)

mu_test_norm = (mu_test - mu_min) / (mu_max - mu_min)
mu_train_LF_norm = (mu_train_LF - mu_min) / (mu_max - mu_min)

In [6]:
for m in range(len(Nlf_models)):
#Loop over the range of the number of basis functions

    print(f"********************  #basis functions = {Nlf_models[m]}  ********************")
    test_mse_HF_list  = []
    test_mse_LF_list  = []
    test_mse_Lin_list = []

    r2_HF_list  = []
    r2_LF_list  = []
    r2_Lin_list = []


    #########################     TRAIN SET      ##########################
    U_lf_train = U_lf_train_full[:,m]

    #########################     TEST SET      ##########################
    U_lf_test = U_lf_test_full[:,m]

    ##########################     NORMALIZATION  ##########################
    #Output
    lfmean = np.mean(U_lf_test)

    U_lf_train = U_lf_train - lfmean
    U_lf_test = U_lf_test - lfmean


    ##########################       FIRST NN: NN_LF     ##########################
    K.clear_session()
    bestLF_params = {'lr' : 0.0255, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam'}

    modelLF = getModel(bestLF_params,'LF')
    
    histLF = modelLF.fit(mu_train_LF_norm, U_lf_train ,epochs=NepoLF,batch_size=Nlf, verbose = 0)
    print('LF NN done')

    ULF = modelLF.predict(mu_test_norm)
    U_LF_list.append(ULF)
    print('\nLF Model:')

    test_mse = np.mean(np.square(U_lf_test - ULF[:,0]))
    test_mse_LF_list.append(test_mse)
    print(f"Test MSE: {test_mse}")

    r_2 = 1 - np.sum(np.square(U_lf_test - ULF[:,0])) / np.sum(np.square(U_lf_test - np.mean(U_lf_test)))
    r2_LF_list.append(r_2)
    print(f"R^2: {r_2}")



    for n_HF in HF_data:
    #loop over the possible numbers of high-fidelity data

        print(f"\n-------  #HF data = {n_HF}  -------")
        start = perf_counter()

        n_HF_txt = str(n_HF) + '.txt'

        #########################     TRAIN SET      #########################
        mu_train_HF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_HF_" + n_HF_txt)
        Nhf = np.size(mu_train_HF) # number of high-fidelity data to TRAIN the NN

        U_hf_train = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Uhf_train_" + n_HF_txt)

        #########################     TEST SET      ##########################
        U_hf_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Uhf_test.txt")

        #permutation = np.random.permutation(len(mu_train_HF))
        #mu_train_HF=mu_train_HF[permutation,:][0:5,:]
        Nlf = np.size(mu_train_HF)  #number of low-fidelity data to TRAIN the NN
        
        #U_hf_train=U_hf_train[permutation][0:5]
        ##########################     NORMALIZATION  ##########################
        #Input
        mu_train_HF_norm = (mu_train_HF - mu_min) / (mu_max - mu_min)

        #Output
        hfmean = np.mean(U_hf_test)

        U_hf_train = U_hf_train - hfmean
        U_hf_test = U_hf_test - hfmean



        ##########################    SECOND NN: NN_Lin    ##########################
        mu_test_help = modelLF.predict(mu_test_norm)[:,0]
              
        
        mu_test_in = np.vstack((mu_test_norm.transpose(),mu_test_help)).transpose() # <- TEST INPUT for the second NN: NN_Lin
        #print(mu_test_in)
        
        mu_train_help = modelLF.predict(mu_train_HF_norm)[:,0] #f_LF(mu_hf_train)
        mu_lin = np.vstack((mu_train_HF_norm.transpose(),mu_train_help)).transpose()  # <- TRAINING INPUT for the second NN: NN_Lin

        bestLin_params = {'lr' : 0.001, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam', 'l2weight' : 0.01}
        modelLin = getModel(bestLin_params,'Hflin')
        histLin = modelLin.fit(mu_lin, U_hf_train, validation_data=(mu_test_in, U_hf_test), epochs=NepoLin,batch_size=Nhf,verbose=0, validation_freq = 50)

        ULin = modelLin.predict(np.vstack((mu_test_norm.transpose(),mu_test_help)).transpose())


        ##########################    THIRD NN: NN_HF    ##########################
        #Input for training NN_HF
        mu_help1 = modelLin.predict(mu_lin)[:,0]
        mu_final = np.vstack((mu_train_HF_norm.transpose() ,mu_train_help, mu_help1)).transpose()

        #Input for testing NN_HF
        mu_test_help1 = modelLin.predict(mu_test_in)[:,0]
        mu_test_final = np.vstack((mu_test_norm.transpose(), mu_test_help, mu_test_help1)).transpose()

        name = '3step'
        MAX_EVAL = 10

        K.clear_session()
        #best paramters obtained by HPO:
        best_params = {'kernel_init': 'glorot_uniform', 'l2weight': 0.0002005990305998077, 'lr': 0.0014207871508967094, 'nodes': 6.0, 'opt': 'Adam'}
        #best_params = {'kernel_init': 'glorot_uniform', 'l2weight': 0.00015338174139229794, 'lr': 0.010752960934369659, 'nodes': 28.0, 'opt': 'Adam'}
        finalModel = getModel(best_params,name) #final model chosen according to the best paramters
        hist = finalModel.fit(mu_final,U_hf_train,validation_data=(mu_test_final, U_hf_test),epochs=NepoHF, batch_size=Nhf, validation_freq=50, verbose=0)

        UHF = finalModel.predict(mu_test_final)
        U_HF_list.append(UHF)

        stop = perf_counter()
        elapsed = stop - start
        print('Elapsed time: ', elapsed)

        ULin = modelLin.predict(np.vstack((mu_test_norm.transpose(),mu_test_help)).transpose())
        U_Lin_list.append(ULin)
        print('\nLin Model:')

        test_mse = np.mean(np.square(U_hf_test - ULin[:,0]))
        test_mse_Lin_list.append(test_mse)
        print(f"Test MSE: {test_mse}")

        r2_Lin = 1 - np.sum(np.square(U_hf_test - ULin[:,0])) / np.sum(np.square(U_hf_test - np.mean(U_hf_test)))
        r2_Lin_list.append(r2_Lin)
        print(f"R^2: {r2_Lin}")


        print('\nHF Model:')
        test_mse = np.mean(np.square(U_hf_test - UHF[:,0]))
        test_mse_HF_list.append(test_mse)
        print(f"Test MSE: {test_mse}")
        r2_HF = 1 - np.sum(np.square(U_hf_test - UHF[:,0])) / np.sum(np.square(U_hf_test - np.mean(U_hf_test)))
        r2_HF_list.append(r2_HF)
        print(f"R^2: {r2_HF}")


    r2_LF_df[str(Nlf_models[m])] = r2_LF_list
    r2_HF_df[str(Nlf_models[m])] = r2_HF_list
    r2_Lin_df[str(Nlf_models[m])] = r2_Lin_list
    mse_LF_df[str(Nlf_models[m])] = test_mse_LF_list
    mse_HF_df[str(Nlf_models[m])] = test_mse_HF_list
    mse_Lin_df[str(Nlf_models[m])] = test_mse_Lin_list

********************  #basis functions = 7  ********************
LF NN done
4/4 [==============================] - 0s 2ms/step

LF Model:
Test MSE: 0.0009785675376389011
R^2: -0.5197591478519059

-------  #HF data = 5  -------
4/4 [==============================] - 0s 2ms/step
Elapsed time:  23.598976099980064
4/4 [==============================] - 0s 0s/step

Lin Model:
Test MSE: 0.0006559425745714788
R^2: -0.01237574472852998

HF Model:
Test MSE: 0.0005987149118145326
R^2: 0.07594890433161916

-------  #HF data = 7  -------
4/4 [==============================] - 0s 6ms/step
Elapsed time:  26.715750600007595
4/4 [==============================] - 0s 0s/step

Lin Model:
Test MSE: 0.0006485225102835436
R^2: -0.0009236856601892995

HF Model:
Test MSE: 0.0006626657387054536
R^2: -0.022752214500310508

-------  #HF data = 9  -------
4/4 [==============================] - 0s 0s/step
Elapsed time:  23.287200399994617
4/4 [==============================] - 0s 4ms/step

Lin Model:
Test MSE: 0.

In [7]:
print(r2_HF_df.round(5))
print(mse_HF_df.round(5))


#########################     SAVE the OUTPUT      ##########################
os.makedirs('Output_new_3p')

r2_HF_df.to_csv('./Output_new_3p/r2_HF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_HF_df.to_csv('./Output_new_3p/mse_HF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')
r2_LF_df.to_csv('./Output_new_3p/r2_LF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_LF_df.to_csv('./Output_new_3p/mse_LF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')
r2_Lin_df.to_csv('./Output_new_3p/r2_Lin_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_Lin_df.to_csv('./Output_new_3p/mse_Lin_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')

          7
5   0.07595
7  -0.02275
9   0.19099
11  0.19595
13 -0.15281
15  0.34906
          7
5   0.00060
7   0.00066
9   0.00052
11  0.00052
13  0.00075
15  0.00042


FileExistsError: [WinError 183] Impossibile creare un file, se il file esiste già: 'Output_new_3p'